# 🤖 Build a Sample Chatbot using Amazon Lex
**K21Academy Hands-On Lab**

Run each cell **in order, top to bottom** to create, configure, build, test and clean up a chatbot on Amazon Lex V2.

> Works on **AWS SageMaker** (credentials come from the SageMaker execution role).

---
## ⚠️ Before you start — one-time IAM setup (2 minutes)

Open **IAM → Roles → `AmazonSageMaker-ExecutionRole-xxxx`** (the role used by your SageMaker domain).

**1. Permissions tab → Add permissions → Create inline policy → JSON** → paste the contents of `Lex_Lab_Permission_Policy.json` → name it `LexLabPolicy` → **Create policy**.

**2. Trust relationships tab → Edit trust policy** → replace everything with the contents of `Lex_Lab_Trust_Policy.json` → **Update policy**.

Step 1 below prints your exact role name and checks these permissions for you. If it shows ❌, fix the policy and re-run Step 1. Don't continue until it shows ✅.

## Step 0 — Install & Import Dependencies

In [1]:
!pip install boto3 --quiet

In [2]:
import boto3
import botocore
import json
import time

REGION = 'us-east-1'

lex_client     = boto3.client('lexv2-models',  region_name=REGION)
runtime_client = boto3.client('lexv2-runtime', region_name=REGION)
sts_client     = boto3.client('sts')
iam_client     = boto3.client('iam')

print('✅ boto3 clients initialized')

✅ boto3 clients initialized


## Step 1 — Get the Execution Role & Check Permissions

This cell finds the SageMaker execution role and checks that it has the Lex permissions. **Continue only when everything shows ✅.**

In [3]:
identity   = sts_client.get_caller_identity()
ACCOUNT_ID = identity['Account']
ROLE_NAME  = identity['Arn'].split('/')[1]   # assumed-role/<ROLE_NAME>/<session>

# Get the full role ARN (including any /service-role/ path)
try:
    ROLE_ARN = iam_client.get_role(RoleName=ROLE_NAME)['Role']['Arn']
except botocore.exceptions.ClientError:
    ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}'

print(f'Account ID : {ACCOUNT_ID}')
print(f'Role name  : {ROLE_NAME}   <-- add the policies to THIS role')
print(f'Role ARN   : {ROLE_ARN}\n')

# Permission check
ok = True
try:
    lex_client.list_bots(maxResults=1)
    print('✅ Lex permissions OK')
except botocore.exceptions.ClientError as e:
    ok = False
    print('❌ Lex permissions missing ->', e.response['Error']['Message'])

try:
    trust = iam_client.get_role(RoleName=ROLE_NAME)['Role']['AssumeRolePolicyDocument']
    if 'lexv2.amazonaws.com' in json.dumps(trust):
        print('✅ Trust policy includes lexv2.amazonaws.com')
    else:
        ok = False
        print('❌ Trust policy is missing lexv2.amazonaws.com (update Trust relationships)')
except botocore.exceptions.ClientError:
    print('⚠️  Could not read the trust policy (iam:GetRole missing). Check it manually in the console.')

print('\n🎉 All good — continue to Step 2' if ok else '\n⛔ Fix the IAM setup (see top of notebook), wait 30s, re-run this cell')

Account ID : 537751920561
Role name  : AmazonSageMaker-ExecutionRole-20261030T123685   <-- add the policies to THIS role
Role ARN   : arn:aws:iam::537751920561:role/service-role/AmazonSageMaker-ExecutionRole-20261030T123685

✅ Lex permissions OK
✅ Trust policy includes lexv2.amazonaws.com

🎉 All good — continue to Step 2


## Step 2 — Create the Bot

In [4]:
BOT_NAME = 'mySampleChatBot'

def find_bot(name):
    res = lex_client.list_bots(filters=[{'name': 'BotName', 'values': [name], 'operator': 'EQ'}])
    for b in res['botSummaries']:
        if b['botName'] == name:
            return b['botId']
    return None

BOT_ID = find_bot(BOT_NAME)
if BOT_ID:
    print(f'ℹ️  Bot already exists, reusing it: {BOT_ID}')
else:
    try:
        BOT_ID = lex_client.create_bot(
            botName=BOT_NAME,
            description='Chatbot for handling greetings and food orders.',
            roleArn=ROLE_ARN,
            dataPrivacy={'childDirected': False},
            idleSessionTTLInSeconds=300
        )['botId']
    except botocore.exceptions.ClientError as e:
        if 'iam:PassRole' in e.response['Error']['Message']:
            raise SystemExit('❌ iam:PassRole denied. Add Lex_Lab_Permission_Policy.json to the role shown in Step 1, wait 30s, re-run.')
        raise

# Wait until the bot is Available
while (s := lex_client.describe_bot(botId=BOT_ID)['botStatus']) != 'Available':
    if s in ('Failed', 'Deleting'):
        raise SystemExit(f'❌ Bot status: {s}')
    print(f'   Bot status: {s} ...'); time.sleep(5)

print(f'✅ Bot ready!  Name: {BOT_NAME}   ID: {BOT_ID}')

   Bot status: Creating ...
✅ Bot ready!  Name: mySampleChatBot   ID: TLPLTF5I7C


## Step 3 — Create a Bot Locale (English)

In [5]:
LOCALE_ID   = 'en_US'
BOT_VERSION = 'DRAFT'

try:
    lex_client.create_bot_locale(
        botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID,
        nluIntentConfidenceThreshold=0.40,
        description='English locale for mySampleChatBot'
    )
except botocore.exceptions.ClientError as e:
    if e.response['Error']['Code'] not in ('ConflictException', 'PreconditionFailedException'):
        raise
    print('ℹ️  Locale already exists, reusing it')

while (s := lex_client.describe_bot_locale(botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID)['botLocaleStatus']) not in ('NotBuilt', 'Built', 'ReadyExpressTesting'):
    if s == 'Failed':
        raise SystemExit('❌ Locale creation failed')
    print(f'   Locale status: {s} ...'); time.sleep(5)

print(f'✅ Bot locale ready: {LOCALE_ID} (confidence threshold: 0.40)')

   Locale status: Creating ...
✅ Bot locale ready: en_US (confidence threshold: 0.40)


## Step 4 — Create Intents (GreetingIntent & OrderFoodIntent)

| Intent | Sample utterances | Response |
|---|---|---|
| GreetingIntent | Hello, Hi, Hai, Hey, Good morning | *Hi, what would you like to order?* |
| OrderFoodIntent | I want to order 1 pizza, I want to order 2 Burgers, … | *Your order has been placed* |

In [6]:
def get_intent_id(name):
    for i in lex_client.list_intents(botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID)['intentSummaries']:
        if i['intentName'] == name:
            return i['intentId']
    return None

def create_intent(name, description, utterances, reply):
    existing = get_intent_id(name)
    if existing:
        print(f'ℹ️  {name} already exists (ID: {existing})')
        return existing
    intent_id = lex_client.create_intent(
        intentName=name,
        description=description,
        sampleUtterances=[{'utterance': u} for u in utterances],
        initialResponseSetting={
            'initialResponse': {
                'messageGroups': [{'message': {'plainTextMessage': {'value': reply}}}],
                'allowInterrupt': True
            }
        },
        botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID
    )['intentId']
    print(f'✅ {name} created (ID: {intent_id}) -> "{reply}"')
    return intent_id

GREETING_INTENT_ID = create_intent(
    'GreetingIntent',
    "Handles user greetings like 'Hello', 'Hi', etc.",
    ['Hello', 'Hi', 'Hai', 'Hey', 'Good morning'],
    'Hi, what would you like to order?'
)

ORDER_INTENT_ID = create_intent(
    'OrderFoodIntent',
    "Handles food orders, e.g., 'I want to order pizza.'",
    ['I want to order 1 pizza', 'I want to order 2 Burgers', 'Can I order food',
     'I would like to order', 'Please take my order'],
    'Your order has been placed'
)

✅ GreetingIntent created (ID: DJXQYIIV9G) -> "Hi, what would you like to order?"
✅ OrderFoodIntent created (ID: KY5CMHDU3D) -> "Your order has been placed"


## Step 5 — Configure FallbackIntent

Responds with **"Sorry, I can't help with that."** for any input the bot doesn't recognise.

In [7]:
FALLBACK_INTENT_ID = get_intent_id('FallbackIntent')

lex_client.update_intent(
    intentId=FALLBACK_INTENT_ID,
    intentName='FallbackIntent',
    parentIntentSignature='AMAZON.FallbackIntent',
    botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID,
    intentClosingSetting={
        'closingResponse': {
            'messageGroups': [{'message': {'plainTextMessage': {'value': "Sorry, I can't help with that."}}}],
            'allowInterrupt': True
        }
    }
)

print(f'✅ FallbackIntent configured (ID: {FALLBACK_INTENT_ID})')

✅ FallbackIntent configured (ID: FALLBCKINT)


## Step 6 — Build the Bot

Starts the build and checks the status every 15 seconds (usually 1–2 minutes).

In [8]:
lex_client.build_bot_locale(botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID)
print('⏳ Building bot...')

while True:
    d = lex_client.describe_bot_locale(botId=BOT_ID, botVersion=BOT_VERSION, localeId=LOCALE_ID)
    status = d['botLocaleStatus']
    print(f'   Status: {status}')
    if status == 'Built':
        print('✅ Bot built successfully!')
        break
    if status in ('Failed', 'NotBuilt'):
        raise SystemExit(f"❌ Build failed: {d.get('failureReasons')}")
    time.sleep(15)

⏳ Building bot...
   Status: Building
   Status: ReadyExpressTesting
   Status: Built
✅ Bot built successfully!


## Step 7 — Create a Bot Version & Alias

In [9]:
print('⏳ Creating bot version...')
BOT_VERSION_NUM = lex_client.create_bot_version(
    botId=BOT_ID,
    botVersionLocaleSpecification={LOCALE_ID: {'sourceBotVersion': 'DRAFT'}}
)['botVersion']
print(f'   Version: {BOT_VERSION_NUM}')

while True:
    try:
        v = lex_client.describe_bot_version(botId=BOT_ID, botVersion=BOT_VERSION_NUM)['botStatus']
        print(f'   Version status: {v}')
        if v == 'Available':
            break
        if v in ('Failed', 'Deleting'):
            raise SystemExit(f'❌ Version failed: {v}')
    except botocore.exceptions.ClientError as e:
        if e.response['Error']['Code'] != 'ResourceNotFoundException':
            raise
    time.sleep(10)

BOT_ALIAS_ID = lex_client.create_bot_alias(
    botAliasName=f'TestAlias{BOT_VERSION_NUM}',
    botId=BOT_ID,
    botVersion=BOT_VERSION_NUM,
    description='Alias for testing mySampleChatBot',
    botAliasLocaleSettings={LOCALE_ID: {'enabled': True}}
)['botAliasId']

while (a := lex_client.describe_bot_alias(botId=BOT_ID, botAliasId=BOT_ALIAS_ID)['botAliasStatus']) != 'Available':
    if a == 'Failed':
        raise SystemExit('❌ Alias creation failed')
    time.sleep(5)

print(f'✅ Alias ready (Alias ID: {BOT_ALIAS_ID}) — ready to test!')

⏳ Creating bot version...
   Version: 1
   Version status: Available
✅ Alias ready (Alias ID: DQCUSTD4HZ) — ready to test!


## Step 8 — Test the Bot

In [10]:
test_messages = [
    'Hello',
    'Hi',
    'I want to order 1 pizza',
    'I want to order 2 Burgers',
    'What is the weather today?',   # -> FallbackIntent
]

print('=' * 55)
print('🧪 CHATBOT TEST RESULTS')
print('=' * 55)

for i, msg in enumerate(test_messages):
    try:
        r = runtime_client.recognize_text(
            botId=BOT_ID, botAliasId=BOT_ALIAS_ID, localeId=LOCALE_ID,
            sessionId=f'test-session-{i}', text=msg
        )
        intent = r['sessionState'].get('intent', {}).get('name', 'N/A')
        msgs = r.get('messages', [])
        reply = msgs[0]['content'] if msgs else '(no response)'
    except Exception as e:
        intent, reply = 'ERROR', str(e)

    print(f'\n👤 User  : {msg}')
    print(f'🎯 Intent: {intent}')
    print(f'🤖 Bot   : {reply}')

print('\n' + '=' * 55)
print('✅ Testing complete!')

🧪 CHATBOT TEST RESULTS

👤 User  : Hello
🎯 Intent: GreetingIntent
🤖 Bot   : Hi, what would you like to order?

👤 User  : Hi
🎯 Intent: GreetingIntent
🤖 Bot   : Hi, what would you like to order?

👤 User  : I want to order 1 pizza
🎯 Intent: OrderFoodIntent
🤖 Bot   : Your order has been placed

👤 User  : I want to order 2 Burgers
🎯 Intent: OrderFoodIntent
🤖 Bot   : Your order has been placed

👤 User  : What is the weather today?
🎯 Intent: FallbackIntent
🤖 Bot   : Sorry, I can't help with that.

✅ Testing complete!


## Step 9 — Cleanup

> ⚠️ Run this only when you're done. It permanently deletes the bot, its alias and versions.
> Afterwards, delete the `LexLabPolicy` inline policy from the execution role and follow the guide to delete the SageMaker space, user profile and domain.

In [2]:
# Standalone cleanup: works even after a kernel restart
import boto3, time

BOT_NAME = 'mySampleChatBot'
lex_client = boto3.client('lexv2-models', region_name='us-east-1')

bots = [b for b in lex_client.list_bots()['botSummaries']
        if b['botName'].startswith(BOT_NAME)]   # also removes test bots like mySampleChatBotss

if not bots:
    print('ℹ️  No mySampleChatBot bots found — nothing to delete.')

for b in bots:
    bot_id = b['botId']
    # Delete aliases first (except the built-in TestBotAlias)
    for a in lex_client.list_bot_aliases(botId=bot_id)['botAliasSummaries']:
        if a['botAliasName'] != 'TestBotAlias':
            lex_client.delete_bot_alias(botId=bot_id, botAliasId=a['botAliasId'], skipResourceInUseCheck=True)
            print(f"🗑️  Alias deleted: {a['botAliasName']}")
    time.sleep(3)
    lex_client.delete_bot(botId=bot_id, skipResourceInUseCheck=True)
    print(f"🗑️  Bot deleted: {b['botName']} ({bot_id})")

print('\n✅ Cleanup complete. All Lex resources removed.')

🗑️  Alias deleted: TestAlias1
🗑️  Bot deleted: mySampleChatBot (TLPLTF5I7C)
🗑️  Bot deleted: mySampleChatBotss (WF5MMFJDIQ)

✅ Cleanup complete. All Lex resources removed.


## Summary

| Step | Action |
|---|---|
| 0 | Installed and imported `boto3` |
| 1 | Found the SageMaker execution role and checked its permissions |
| 2 | Created bot `mySampleChatBot` |
| 3 | Configured English locale (`en_US`) |
| 4 | Created `GreetingIntent` and `OrderFoodIntent` |
| 5 | Configured `FallbackIntent` |
| 6 | Built the bot |
| 7 | Created bot version and alias |
| 8 | Tested the bot with sample messages |
| 9 | Cleaned up all resources |

---
*Copyright © K21Academy | All Rights Reserved*